# PokeAPI Data Cleaning

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

## Global Variables

In [2]:
DATA_PATH   = Path(Path.cwd().parent.parent, 'data')

BRONZE_PATH = Path(DATA_PATH / '1-bronze')
SILVER_PATH = Path(DATA_PATH / '2-silver')

ABILITY_PATH         = Path(BRONZE_PATH / "PokeAPI" / "ability")
ITEM_PATH            = Path(BRONZE_PATH / "PokeAPI" / "item")
MOVE_PATH            = Path(BRONZE_PATH / "PokeAPI" / "move")
NATURE_PATH          = Path(BRONZE_PATH / "PokeAPI" / "nature")
POKEMON_PATH         = Path(BRONZE_PATH / "PokeAPI" / "pokemon")
POKEMON_SPECIES_PATH = Path(BRONZE_PATH / "PokeAPI" / "pokemon-species")
TYPE_PATH            = Path(BRONZE_PATH / "PokeAPI" / "type")

SILVER_POKEAPI_PATH  = Path(SILVER_PATH / "PokeAPI")

GEN_DICT_NAME = {
    "generation-i": 1,
    "generation-ii": 2,
    "generation-iii": 3,
    "generation-iv": 4,
    "generation-v": 5,
    "generation-vi": 6,
    "generation-vii": 7,
    "generation-viii": 8,
    "generation-ix": 9,    
}

## Helpers

In [3]:
def create_dataframe(dir_path: Path) -> pd.DataFrame:
    data_list = []
    
    for file_path in dir_path.iterdir():
        with open(file_path, 'r') as f:
            data = json.load(f)
            data_list.append(data)

    return pd.DataFrame(data_list)

In [4]:
def normalize_json_column(df:pd.DataFrame, column_name: str) -> pd.DataFrame:
    try:
        new_columns = pd.json_normalize(df[column_name].to_list())
        for i in new_columns:
            df[f"{column_name}_{i}"] = new_columns[i]

        df = df.drop(columns=column_name)
        
        return df

    except KeyError:
        print(f"Non Existing Column with name: {column_name}")
        return df


In [5]:
def normalize_json_list_column(df:pd.DataFrame, column_name: str) -> pd.DataFrame:
    try:
        df = df.explode(column_name, ignore_index=True)
        df = normalize_json_column(df, column_name)

        return df
        

    except KeyError:
        print(f"Non Existing Column with name: {column_name}")
        return df

In [8]:
# PokeAPI markup "[label]{type:id}" becomes "label" (or the id without dashes when label is empty)
def clean_text(series: pd.Series) -> pd.Series:
    series = series.str.replace(r'\[([^\]]*)\]\{[^:}]+:([^}]+)\}', lambda m: m.group(1) or m.group(2).replace('-', ' '), regex=True)
    return series.str.replace(r'\s+', ' ', regex=True).str.strip()

## Creating DataFrames and Preparing the Data

### Ability:

In [9]:
df = create_dataframe(ABILITY_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 374


,name,is_main_series,generation,names,effect_entries,effect_changes,flavor_text_entries,pokemon
id,,,,,,,,
1,stench,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'あくしゅう', 'language': {'name': 'ja-hr...",[{'effect': 'Les capacités offensives de ce Po...,"[{'version_group': {'name': 'black-white', 'ur...","[{'flavor_text': 'Repousse POKéMON sauvage.', ...","[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
2,drizzle,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'あめふらし', 'language': {'name': 'ja-hr...",[{'effect': 'La météo devient pluvieuse quand ...,[],"[{'flavor_text': 'Summons rain in battle.', 'l...","[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
3,speed-boost,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'かそく', 'language': {'name': 'ja-hrkt...",[{'effect': 'La Vitesse de ce Pokémon augmente...,[],"[{'flavor_text': 'Améliore la VITESSE.', 'lang...","[{'is_hidden': False, 'slot': 1, 'pokemon': {'..."
4,battle-armor,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'カブトアーマー', 'language': {'name': 'ja-...",[{'effect': 'Les capacités ne peuvent pas infl...,[],[{'flavor_text': 'Bloque les coups critiques.'...,"[{'is_hidden': True, 'slot': 3, 'pokemon': {'n..."
5,sturdy,True,"{'name': 'generation-iii', 'url': 'https://pok...","[{'name': 'がんじょう', 'language': {'name': 'ja-hr...","[{'effect': 'Quand ce Pokémon a tous ses PV, n...","[{'version_group': {'name': 'black-white', 'ur...","[{'flavor_text': 'Negates 1-hit KO attacks.', ...","[{'is_hidden': False, 'slot': 2, 'pokemon': {'..."


In [10]:
df['is_main_series'].value_counts()

is_main_series
True     314
False     60
Name: count, dtype: int64

In [11]:
# Recreate for not popping errors when running this cell 2+ times
df = create_dataframe(ABILITY_PATH)

# Dropping columns that will not be used
df = df.drop(columns=['names', 'pokemon', 'flavor_text_entries'])

# WARNING: Since gen 9 is the current gen, I can drop this column easily, if you are working in a gen that is not the current gen: ADAPT THIS CODE!
df = df.drop(columns='effect_changes')

# Main Series only, deleting != True:
df = df[df['is_main_series']].reset_index(drop=True)
df = df.drop(columns='is_main_series')

df = normalize_json_column(df, 'generation')
df = df.drop(columns=['generation_url'])

effects = normalize_json_list_column(df[['id', 'effect_entries']], 'effect_entries')
effects = effects.loc[effects['effect_entries_language.name'] == 'en', ['id', 'effect_entries_effect', 'effect_entries_short_effect']]

df = df.drop(columns=['effect_entries']).merge(effects, on='id')

# Needs to be last step, otherwise filters won't work and dropping the non-main series ill also break
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

N of Rows: 314


,name,generation_name,effect_entries_effect,effect_entries_short_effect
id,,,,
1,stench,generation-iii,This Pokémon's damaging moves have a 10% chanc...,Has a 10% chance of making target Pokémon flin...
2,drizzle,generation-iii,The weather changes to rain when this Pokémon ...,Summons rain that lasts indefinitely upon ente...
3,speed-boost,generation-iii,This Pokémon's Speed rises one stage after eac...,Raises Speed one stage after each turn.
4,battle-armor,generation-iii,Moves cannot score critical hits against this ...,Protects against critical hits.
5,sturdy,generation-iii,"When this Pokémon is at full HP, any hit that ...","Prevents being KOed from full HP, leaving 1 HP..."


In [12]:
df['generation_name'].value_counts()

generation_name
generation-iii     76
generation-iv      47
generation-ix      47
generation-vii     42
generation-v       41
generation-viii    34
generation-vi      27
Name: count, dtype: int64

In [13]:
df['generation'] = df['generation_name'].map(GEN_DICT_NAME)
df = df.drop(columns='generation_name')

df = df.rename(columns={"effect_entries_effect": "effect"})
df = df.rename(columns={"effect_entries_short_effect": "short_effect"})

df['generation'].value_counts()

generation
3    76
4    47
9    47
7    42
5    41
8    34
6    27
Name: count, dtype: int64

In [14]:
df.head()

,name,effect,short_effect,generation
id,,,,
1,stench,This Pokémon's damaging moves have a 10% chanc...,Has a 10% chance of making target Pokémon flin...,3
2,drizzle,The weather changes to rain when this Pokémon ...,Summons rain that lasts indefinitely upon ente...,3
3,speed-boost,This Pokémon's Speed rises one stage after eac...,Raises Speed one stage after each turn.,3
4,battle-armor,Moves cannot score critical hits against this ...,Protects against critical hits.,3
5,sturdy,"When this Pokémon is at full HP, any hit that ...","Prevents being KOed from full HP, leaving 1 HP...",3


In [15]:
# Data quality checks
print(f"Duplicated ids:   {df.index.duplicated().sum()}")
print(f"Duplicated names: {df['name'].duplicated().sum()}")
print(f"Duplicated rows:  {df.duplicated().sum()}")

print("----------\nNulls per column:")
print(df.isna().sum())

text_cols = df.select_dtypes('str')

print("----------\nEmpty strings per column:")
print((text_cols.apply(lambda c: c.str.strip()) == '').sum())

print("----------\nTexts with newlines or markup ([...]{...}):")
print(text_cols.apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

Duplicated ids:   0
Duplicated names: 0
Duplicated rows:  0
----------
Nulls per column:
name            0
effect          0
short_effect    0
generation      0
dtype: int64
----------
Empty strings per column:
name            0
effect          0
short_effect    0
dtype: int64
----------
Texts with newlines or markup ([...]{...}):
name              0
effect          131
short_effect      0
dtype: int64


In [16]:
# Plain text only: markup and newlines are removed from the effect texts, overwriting the original columns
df['effect'] = clean_text(df['effect'])
df['short_effect'] = clean_text(df['short_effect'])

print("Texts with newlines or markup after cleaning:")
print(df[['effect', 'short_effect']].apply(lambda c: c.str.contains(r'\n|\f|\]\{', regex=True)).sum())

Texts with newlines or markup after cleaning:
effect          0
short_effect    0
dtype: int64


In [17]:
# Data types:
df['generation'] = df['generation'].astype('int8')

print(df.dtypes)
print(f"----------\nIndex dtype: {df.index.dtype}")
print(f"Names not in kebab-case: {(~df['name'].str.fullmatch(r'[a-z0-9-]+')).sum()}")

name             str
effect           str
short_effect     str
generation      int8
dtype: object
----------
Index dtype: int64
Names not in kebab-case: 0


In [18]:
# Text lengths, to spot truncated or broken texts
lengths = df[['effect', 'short_effect']].apply(lambda c: c.str.len())
lengths.describe()

,effect,short_effect
count,314.000000,314.000000
mean,202.156051,64.159236
std,214.922122,28.943375
min,27.000000,15.000000
25%,83.250000,47.000000
50%,126.000000,58.500000
75%,240.750000,76.000000
max,1538.000000,228.000000


In [ ]:
# Shortest and longest effects
df.loc[lengths.sort_values('effect').index].iloc[[0, 1, 2, -3, -2, -1]]

,name,effect,short_effect,generation
id,,,,
39,inner-focus,This Pokémon cannot flinch.,Prevents flinching.,3
169,fur-coat,Halves damage from physical attacks.,Halves damage from physical attacks.,6
247,ripen,Ripens Berries and doubles their effect.,Doubles the effect of berries.,8
104,mold-breaker,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,4
163,turboblaze,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,5
164,teravolt,This Pokémon's moves completely ignore abiliti...,Bypasses targets' abilities if they could hind...,5


In [20]:
SILVER_POKEAPI_PATH.mkdir(parents=True, exist_ok=True)

df.reset_index().to_parquet(SILVER_POKEAPI_PATH / 'ability.parquet', index=False)

check = pd.read_parquet(SILVER_POKEAPI_PATH / 'ability.parquet').set_index('id')
print(f"Rows: {len(check)} | Same dtypes: {check.dtypes.equals(df.dtypes)} | Same data: {check.equals(df)}")

Rows: 314 | Same dtypes: True | Same data: True


### Item:

In [ ]:
df = create_dataframe(ITEM_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

### Move:

In [ ]:
df = create_dataframe(MOVE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

### Nature:

In [ ]:
df = create_dataframe(NATURE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

### Type:

In [ ]:
df = create_dataframe(TYPE_PATH)
df = df.set_index('id')

print(f"N of Rows: {len(df)}")
df.head()

### Pokemon + Pokemon_Species:

In [ ]:
df_species = create_dataframe(POKEMON_SPECIES_PATH)

df_species = df_species[['id', 'is_legendary', 'is_mythical', 'generation']].rename(columns={'id': 'species_id'})
df_species = normalize_json_column(df_species, 'generation')
df_species = df_species.drop(columns='generation_url')
df_species['generation'] = df_species['generation_name'].map(GEN_DICT_NAME)
df_species = df_species.drop(columns='generation_name')

print(f"N of Rows: {len(df_species)}")
df_species.head()

In [ ]:
df_pokemon = create_dataframe(POKEMON_PATH)
df_pokemon['species_id'] = df_pokemon['species'].str.get('url').str.extract(r'/(\d+)/$')[0].astype(int)
df_pokemon = df_pokemon.drop(columns='species')

print(f"N of Rows: {len(df_pokemon)}")
df_pokemon.head()

In [ ]:
df = df_pokemon.merge(df_species, on='species_id', how='left', validate='many_to_one')

print(f"N of Rows: {len(df)}")
df.head()